In [ ]:
import os

print("INPUT ROOT:")
print(os.listdir("/kaggle/input"))

print("\nDATASETS:")
print(os.listdir("/kaggle/input/datasets"))

print("\nSAUTKIN DATASETS:")
print(os.listdir("/kaggle/input/datasets/sautkin"))

## Clean the Working Directory

In [ ]:
import shutil
import os

for item in os.listdir("/kaggle/working"):
    path = os.path.join("/kaggle/working", item)

    if os.path.isdir(path):
        shutil.rmtree(path)
    else:
        os.remove(path)

In [ ]:
!rm -rf LSNET-advanced
!git clone -b proposal_8 https://github.com/Param45/LSNET-advanced.git
%cd LSNET-advanced
!ls

In [ ]:
import torch
print(torch.__version__)

In [ ]:
!pip install -q timm fvcore wandb
!pip install timm==0.5.4 einops==0.4.1

In [ ]:
from kaggle_config import IMAGENET_PATH

print(IMAGENET_PATH)

In [ ]:
import os

for ds in [
    "imagenet1k0",
    "imagenet1k1",
    "imagenet1k2",
    "imagenet1k3",
    "imagenet1kvalid"
]:
    path = f"/kaggle/input/datasets/sautkin/{ds}"
    print(ds, len(os.listdir(path)))

In [ ]:
!python main.py --help

## Verify Dataset Loading (40% fraction)

Proposal 8 uses only 40% of the ImageNet-1k training data.
The `--dataset-fraction 0.4` flag is passed to `main.py` which
passes it to `build_dataset`, which randomly subsamples 40% of
the training files (fixed seed=42) before building the dataset object.
Validation set is always loaded in full.

In [ ]:
from data.datasets import build_dataset

In [ ]:
from types import SimpleNamespace
import time

args = SimpleNamespace(
    data_set="IMNET",
    data_path="/kaggle/input/datasets/sautkin",
    input_size=224,
    color_jitter=0.4,
    aa='rand-m9-mstd0.5-inc1',
    train_interpolation='bicubic',
    reprob=0.25,
    remode='pixel',
    recount=1,
    finetune='',
    inat_category='name',
    dataset_fraction=0.4
)

start = time.time()

dataset_train, n_classes = build_dataset(
    is_train=True,
    args=args
)

print("Classes:", n_classes)
print("Samples:", len(dataset_train))
print("Time:", time.time() - start)

In [ ]:
start = time.time()

dataset_val, n_classes = build_dataset(
    is_train=False,
    args=args
)

print("Classes:", n_classes)
print("Samples:", len(dataset_val))
print("Time:", time.time() - start)

In [ ]:
!python kaggle_run.py --help

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

## Verify Checkpoint Loading — Proposal 8 (Shifted-Window SKA)

**What to expect:**
- `ShiftedSKA` has **zero new parameters** — checkpoint loads with `strict=True`.
- `missing=[]`, `unexpected=[]`.
- Every `LSConv` block with `depth % 4 == 3` uses `ShiftedSKA`; the rest use the normal `SKA`.
- No `group_se` keys in the state dict.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/LSNET-advanced')

import torch
from timm.models import create_model
import model  # registers lsnet_t

m = create_model('lsnet_t', use_shifted_ska=True)

ckpt = torch.load('/kaggle/working/LSNET-advanced/pretrain/lsnet_t.pth', map_location='cpu')
state = ckpt['model']

missing, unexpected = m.load_state_dict(state, strict=True)
print("Missing:", missing)       # expected []
print("Unexpected:", unexpected)  # expected []

## Run Fine-Tuning — Proposal 8 (Shifted-Window SKA, 40% Dataset)

**What this does:**
- Loads the pretrained `lsnet_t.pth` weights with `strict=True` — ShiftedSKA adds zero new parameters so all keys match exactly.
- Every `LSConv` block at `depth % 4 == 3` uses a shifted-window SKA (cyclic roll by 1 pixel before the 3×3 aggregation, then unroll after).
- Adjacent blocks (regular + shifted) together tile the spatial neighbourhood.
- BN running stats for shifted blocks adapt within 2–5 epochs.
- Trains for 14 epochs with 2 warmup epochs on 40% of ImageNet-1k.

In [ ]:
# Fine-tune for 14 epochs for Proposal 8 with 40% dataset

%cd /kaggle/working/LSNET-advanced
!python kaggle_run.py --action train_clf --model lsnet_t --finetune /kaggle/working/LSNET-advanced/pretrain/lsnet_t.pth --epochs 14 --warmup-epochs 2 --lr 5e-5 --use-shifted-ska --dataset-fraction 0.4
